<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Lesson 1 精华 — Build the Email Assistant

**一句话定位**:**外层 workflow + 内层 agent** 的组合范式 —— router 用 `Command` 串子图,agent 用 `bind_tools` + 条件边手搓 ReAct。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**这节课在整门课的位置**:Module 2 / 6 —— **真正搭出 agent**(后面 M3 测它、M4 加 HITL、M5 加记忆、M6 部署)。

**4 大组件 + 一个关键设计决策**:

- `State`(扩展 `MessagesState`)· `triage_router`(workflow)· `response_agent`(subgraph)· `should_continue`(条件边)
- **决策**:**workflow 包 agent** —— 三选一的 triage 用 workflow 写死,开放的 response 用 agent 自由发挥

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🏗️ 架构总览 — 数据流

</div>

```
Email Input ──▶ triage_router(workflow,with_structured_output)
                         │
          ┌──────────────┼──────────────┐
          ▼ respond      ▼ notify       ▼ ignore
     response_agent     END             END
     (subgraph)
     ├─ llm_call ◀──────────┐
     ├─ tool_handler ───────┘
     └─ should_continue ─▶ END(when Done called)
```

| 组件 | 角色 | 关键 API |
|------|------|----------|
| **`State`** | 扩展 `MessagesState`,加 `email_input` + `classification_decision` | `class State(MessagesState):` |
| **`triage_router`** | 分类邮件 → 跳分支 + 写 state | `with_structured_output(RouterSchema)` + `Command(goto, update)` |
| **`response_agent`** | ReAct 循环:llm_call ↔ tool_handler | `bind_tools(tools, tool_choice="any")` |
| **`should_continue`** | 条件边:见到 `Done` 工具名 → END | 检查 `last.tool_calls[i]["name"]` |
| **4 个 tools** | `write_email` · `schedule_meeting` · `check_calendar_availability` · `Done`(class) | `@tool` 装饰函数 + Pydantic class |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔀 Triage Router — workflow 用 Command 串子图

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">class RouterSchema(BaseModel):
    reasoning: str       = Field(description="Step-by-step reasoning.")
    classification: Literal["ignore", "respond", "notify"]

llm_router = llm.<span style="background:#3d3414; color:#fde047; padding:0 2px;">with_structured_output(RouterSchema)</span>

def triage_router(state: State) -> Command[Literal["response_agent", "__end__"]]:
    result = llm_router.invoke([sys_prompt, user_prompt])
    if result.classification == "respond":
        return <span style="background:#3d3414; color:#fde047; padding:0 2px;">Command(goto="response_agent", update={
            "messages": [{"role":"user", "content": format_email(...)}],   # ← 给子图喂第一条消息
            "classification_decision": result.classification,
        })</span>
    elif result.classification in ("ignore", "notify"):
        return Command(goto=END, update={"classification_decision": result.classification})
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 关键洞察:`update={"messages":[...]}` 是父子图之间的「**信件**」**

Triage router(父图)往 `state["messages"]` 塞一条 `"Respond to the email: ..."`,然后 `goto="response_agent"`(子图)。子图启动时直接看到这条消息,**当成用户提问** 开始 ReAct。

**MessagesState 的 `add_messages` reducer** 在这里发挥作用:不覆盖,而是 append。所以子图后续累积的 tool_calls / ToolMessages 也都留在同一个 messages 列表里,父图最后能完整看到整条 trace。

</div>

📎 Command vs Edges 的对比详见 [0_b_⭐️_edges_vs_Command.ipynb](./0_b_⭐️_edges_vs_Command.ipynb)。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🤖 Response Agent — 三件套手搓 ReAct

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python"># 1️⃣ llm_call — 让 LLM 决定调哪个工具
llm_with_tools = llm.bind_tools(tools, <span style="background:#3d3414; color:#fde047; padding:0 2px;">tool_choice="any"</span>)   # 强制至少调一个

def llm_call(state):
    return {"messages": [llm_with_tools.invoke([sys] + state["messages"])]}

# 2️⃣ tool_handler — 真正执行 tool_calls
def tool_handler(state):
    result = []
    for tc in state["messages"][-1].tool_calls:
        observation = tools_by_name[tc["name"]].invoke(tc["args"])
        result.append({"role":"tool", "content":observation, "tool_call_id":tc["id"]})
    return {"messages": result}

# 3️⃣ should_continue — 条件边,检测 Done 信号
def should_continue(state) -> Literal["tool_handler", "__end__"]:
    for tc in state["messages"][-1].tool_calls:
        if tc["name"] == <span style="background:#3d3414; color:#fde047; padding:0 2px;">"Done"</span>:
            return END                  # ← 信号型工具的运行时配合
        return "tool_handler"
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 关键洞察:`Done` 的真正归宿在 `should_continue`,不在 `tool_handler`**

`Done` 是 Pydantic class,**没有 @tool 装饰的函数体**。`tool_handler` 即便对它 `.invoke(...)` 也基本是 no-op(只产生空 ToolMessage)。它的全部价值 = **`should_continue` 检测 `tc["name"] == "Done"` → 直接 return END**。

👉 **信号型工具 = LLM 的「表达意图」 + 条件边的「字符串匹配」配对工作**。

📎 工具类型详见 [0_c_⭐️_工具类型大全.ipynb](./0_c_⭐️_工具类型大全.ipynb)。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔗 组装:外层 graph 把 router 和 agent 拼起来

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python"># Step 1 — 先编译 response agent(子图)
agent_workflow = StateGraph(State)
agent_workflow.add_node("llm_call", llm_call)
agent_workflow.add_node("tool_handler", tool_handler)
agent_workflow.add_edge(START, "llm_call")
agent_workflow.add_conditional_edges("llm_call", should_continue,
                                     {"tool_handler": "tool_handler", END: END})
agent_workflow.add_edge("tool_handler", "llm_call")
<span style="background:#3d3414; color:#fde047; padding:0 2px;">agent = agent_workflow.compile()</span>     # ← 编译成可作为 node 用的子图

# Step 2 — 把子图当 node 塞进外层图
overall = (StateGraph(State)
    .add_node(triage_router)                    # workflow 部分
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">.add_node("response_agent", agent)</span>          # ← 子图作为 node
    .add_edge(START, "triage_router")
).compile()
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 一个图可以作 node 嵌进另一个图**

`agent_workflow.compile()` 返回的 `agent` 对象,**本身就是个可调用的 graph**,也满足 LangGraph 的 node 接口(接 state,返回 state 更新)。所以可以直接 `add_node("response_agent", agent)` 当成子图嵌入。

这是 LangGraph **递归组合性** 的核心 —— 同样的「nodes + edges + state」原语,既能搭单个 agent,也能搭 agent 之上的 workflow,**层数想加多深都行**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ⚠️ 4 个易踩坑

</div>

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

- ❌ `triage_router` 的 `update={"messages":[...]}` 忘了写 → **子图收不到「要回什么邮件」**,response_agent 启动时直接懵
- ❌ `bind_tools(..., tool_choice="any")` 漏写默认值 → LLM 可能输出纯文本不调工具,`should_continue` 访问空 `tool_calls` 抛错
- ❌ `should_continue` 忘了把 `END` 加进路由 dict(`{"tool_handler": "tool_handler", END: END}`) → 命中 END 时找不到对应 node 报错
- ❌ 子图编译前就 `.add_node("response_agent", agent)` → 拿到的是未 compile 的 `StateGraph`,**会报「不是 Runnable」**

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**Build 的核心范式 = workflow 包 agent**:

- **能在白板画清的部分**(三选一 triage)→ workflow 写死,用 `Command(goto, update)` **一行同时跳转+喂消息**
- **开放的部分**(怎么回信)→ agent 自由发挥,用 `bind_tools` + `should_continue` **检测 `Done` 信号** 来终止
- **嵌套用**:子图 `.compile()` 后直接 `add_node("name", agent)`,层级递归没限制

🎯 **下一步**:用 [2_z_⭐️_本节精华_Evaluate.ipynb](./2_z_⭐️_本节精华_Evaluate.ipynb) 学怎么测它。

</div>